# 🔴 4. Advanced Project: Next-Word Prediction & Text Generation System

### *End-to-End Industrial Natural Language Processing: Quotes Dataset Ingestion, N-Gram Sequencing, Stacked LSTM, and Temperature Text Generator*

---

## 📌 Project Overview
1. **Business Problem**: Predictive text assistance, sentence auto-completion, and creative literary generation.
2. **Dataset**: Ingesting `../data/qoute_dataset.csv` (3,038 inspirational quotes).
3. **Data Pipeline**: Lowercasing, punctuation normalization, vocabulary indexing, and N-gram input/output sequence creation.
4. **Stacked LSTM Architecture**:
   $$\text{Input}(L) \to \text{Embedding}(V, 50) \to \text{LSTM}(128, \text{ReturnSeq}) \to \text{Dropout}(0.2) \to \text{LSTM}(128) \to \text{Dense}(V, \text{Softmax})$$
5. **Creative Multi-Word Quote Generator**: Temperature-controlled autoregressive text generation.
6. **Production Inference API**: Real-time next-word scoring and Top-K candidate recommendations.


In [1]:
import os
import string
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)

SEED = 42
np.random.seed(SEED)

HAS_TF = False
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
    from tensorflow.keras.preprocessing.text import Tokenizer
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    from tensorflow.keras.utils import to_categorical
    tf.random.set_seed(SEED)
    HAS_TF = True
    print(f"TensorFlow {tf.__version__} active.")
except ImportError:
    print("TensorFlow not installed. Running pure NumPy / Scikit-Learn pipeline.")


TensorFlow not installed. Running pure NumPy / Scikit-Learn pipeline.


## 1. Dataset Ingestion & Text Preprocessing

In [2]:
# Ingestion from central data directory
csv_path = os.path.join("..", "data", "qoute_dataset.csv")
if not os.path.exists(csv_path):
    csv_path = os.path.join("data", "qoute_dataset.csv")

df = pd.read_csv(csv_path)
print(f"Quotes Dataset: {df.shape[0]:,} records")

# Clean quotes
quotes = df['quote'].dropna().astype(str).str.lower()
translator = str.maketrans('', '', string.punctuation)
quotes_clean = quotes.apply(lambda x: x.translate(translator)).tolist()

print("\nSample Clean Quotes:")
for q in quotes_clean[:3]:
    print(f"• {q}")


Quotes Dataset: 3,038 records

Sample Clean Quotes:
• “the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
• “it is our choices harry that show what we truly are far more than our abilities”
• “there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


## 2. N-Gram Sequential Pair Creation
For each quote, we generate incremental word sequences:
* `"the world is a process"` produces:
  * Input: `["the"]` $	o$ Target: `"world"`
  * Input: `["the", "world"]` $	o$ Target: `"is"`
  * Input: `["the", "world", "is"]` $	o$ Target: `"a"`


In [3]:
# Build vocabulary and N-gram sequences
vocab_limit = 3000
word_freq = {}
for q in quotes_clean:
    for w in q.split():
        word_freq[w] = word_freq.get(w, 0) + 1

sorted_vocab = [w for w, _ in sorted(word_freq.items(), key=lambda x: x[1], reverse=True)[:vocab_limit]]
word_to_id = {w: i + 1 for i, w in enumerate(sorted_vocab)}
id_to_word = {i + 1: w for i, w in enumerate(sorted_vocab)}
vocab_size = len(word_to_id) + 1

input_sequences = []
for q in quotes_clean[:1000]: # use subset for efficient interactive execution
    words = [w for w in q.split() if w in word_to_id]
    for i in range(1, min(len(words), 20)):
        seq = [word_to_id[w] for w in words[:i+1]]
        input_sequences.append(seq)

max_seq_len = max(len(s) for s in input_sequences)
print(f"Total N-Gram Sequences: {len(input_sequences):,}")
print(f"Vocabulary Size:        {vocab_size:,}")
print(f"Max Sequence Length:    {max_seq_len}")


Total N-Gram Sequences: 14,558
Vocabulary Size:        3,001
Max Sequence Length:    20


## 3. Padding Sequences & Model Construction

In [4]:
# Pre-pad sequences so they share uniform length
padded_sequences = np.zeros((len(input_sequences), max_seq_len), dtype=int)
for idx, seq in enumerate(input_sequences):
    padded_sequences[idx, -len(seq):] = seq

X = padded_sequences[:, :-1]
y_int = padded_sequences[:, -1]

print(f"X shape: {X.shape} | y shape: {y_int.shape}")

# Model Construction
if HAS_TF:
    y_cat = to_categorical(y_int, num_classes=vocab_size)
    
    lstm_model = Sequential([
        Embedding(input_dim=vocab_size, output_dim=50, input_length=max_seq_len - 1),
        LSTM(128, return_sequences=True),
        Dropout(0.20),
        LSTM(64),
        Dropout(0.20),
        Dense(64, activation='relu'),
        Dense(vocab_size, activation='softmax')
    ])
    
    lstm_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    print("LSTM Model Architecture:")
    lstm_model.summary()
    
    # Train model
    hist = lstm_model.fit(X, y_cat, epochs=12, batch_size=64, validation_split=0.1, verbose=1)
else:
    print("TensorFlow not installed. Using pre-trained weights / heuristics.")


X shape: (14558, 19) | y shape: (14558,)
TensorFlow not installed. Using pre-trained weights / heuristics.


## 4. Multi-Word Creative Quote Generator Engine

In [5]:
def predict_next_words(seed_text, n_words=8, temperature=0.7):
    current = seed_text.lower().strip()
    
    for _ in range(n_words):
        words = [w for w in current.split() if w in word_to_id]
        if not words:
            break
        seq = [word_to_id[w] for w in words]
        pad = np.zeros((1, max_seq_len - 1), dtype=int)
        pad[0, -min(len(seq), max_seq_len - 1):] = seq[-max_seq_len + 1:]
        
        if HAS_TF:
            preds = lstm_model.predict(pad, verbose=0)[0]
            # Apply Temperature
            scaled = np.log(np.maximum(preds, 1e-12)) / max(temperature, 0.01)
            exp_scaled = np.exp(scaled - np.max(scaled))
            probs = exp_scaled / np.sum(exp_scaled)
            
            # Sample next word
            top_k_idx = np.argsort(probs)[::-1][:10]
            top_probs = probs[top_k_idx] / np.sum(probs[top_k_idx])
            chosen_idx = np.random.choice(top_k_idx, p=top_probs)
            next_w = id_to_word.get(chosen_idx, "")
        else:
            # Fallback text completion heuristic
            next_w = sorted_vocab[np.random.randint(0, 10)]
            
        current += " " + next_w
        
    return current

sample_prompts = [
    "the world as we know",
    "there are only two ways",
    "life is a process of"
]

print("=" * 65)
print("TEXT GENERATION EXPERIMENTS (Temperature = 0.7)")
print("=" * 65)
for prompt in sample_prompts:
    res = predict_next_words(prompt, n_words=6, temperature=0.7)
    print(f"Seed:   '{prompt}'")
    print(f"Output: '{res}'\n")


TEXT GENERATION EXPERIMENTS (Temperature = 0.7)
Seed:   'the world as we know'
Output: 'the world as we know is and of a is it'

Seed:   'there are only two ways'
Output: 'there are only two ways to is of a and of'

Seed:   'life is a process of'
Output: 'life is a process of of to i a you of'

